# Day 04 上午课堂练习：电商用户数据清洗与预处理

**主数据文件：** E Commerce Dataset.xlsx（使用 E Comm 工作表）

**提交要求：** 完成所有 TODO，运行全部单元后提交本 Notebook 与清洗后的 CSV 文件。

## 学习目标

- 检查字段类型、缺失值和重复记录；
- 使用中位数填补数值缺失；
- 统一类别字段的同义取值；
- 使用统计规则和业务规则检查候选异常值；
- 导出清洗后的数据。


---
## 1. 读取数据

如报找不到文件，请修改候选路径或 DATA_PATH。


In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.2f}")

candidates = [
    Path("E Commerce Dataset.xlsx"),
    Path("../data/E Commerce Dataset.xlsx"),
    Path("data/E Commerce Dataset.xlsx"),
]
DATA_PATH = next((path for path in candidates if path.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError("未找到 E Commerce Dataset.xlsx，请修改 DATA_PATH。")

df = pd.read_excel(DATA_PATH, sheet_name="E Comm")
print(f"读取文件：{DATA_PATH}")
print(f"数据形状：{df.shape[0]} 行，{df.shape[1]} 列")
df.head()


### 任务 1：理解数据

运行下一单元，并以注释回答：

1. 一行数据代表什么？
2. 哪个字段是用户唯一标识？
3. 哪个字段可作为用户流失分析的目标变量？


In [ ]:
df.info()

# 答案：
# 1. 一行数据代表一名电商用户的完整信息记录
# 2. CustomerID 是用户唯一标识
# 3. Churn 字段可作为用户流失分析的目标变量，值为1表示已流失


---
## 2. 数据质量检查

数据清洗前，先检查缺失值和重复值。


### 任务 2：生成缺失值报告

创建 missing_report，包含“缺失数量”和“缺失比例”两列；按缺失数量降序排列。缺失比例用百分比表示，保留两位小数。


In [ ]:
missing_report = pd.DataFrame({
    "缺失数量": df.isna().sum(),
    "缺失比例": (df.isna().mean() * 100).round(2)
}).sort_values(by="缺失数量", ascending=False)

display(missing_report)


### 任务 3：检查重复记录

分别统计完全重复行数与 CustomerID 重复数量。思考：CustomerID 重复时，能否直接删除？


In [ ]:
duplicate_rows = df.duplicated().sum()
duplicate_customer_ids = df["CustomerID"].duplicated().sum()

print("完全重复行数：", duplicate_rows)
print("CustomerID 重复数量：", duplicate_customer_ids)

# 思考：用户 ID 重复时，不能直接删除，因为可能是用户信息的多次记录或不同时间点的快照，
# 需要进一步分析重复原因，如是否包含不同时间的数据或是否为数据录入错误。


---
## 3. 缺失值处理

本练习对数值型缺失统一采用中位数填充。缺失不等于 0，例如订单数缺失并不能说明用户没有下单。


### 任务 4：用中位数填补数值缺失

请对下列字段逐列使用中位数填充，随后检查剩余缺失值。


In [ ]:
numeric_missing_cols = [
    "Tenure",
    "WarehouseToHome",
    "HourSpendOnApp",
    "OrderAmountHikeFromlastYear",
    "CouponUsed",
    "OrderCount",
    "DaySinceLastOrder",
]

for col in numeric_missing_cols:
    median_value = df[col].median()
    df[col] = df[col].fillna(median_value)

print(df[numeric_missing_cols].isna().sum())


### 思考题

什么时候不适合用中位数填充？写出一种场景及更合适的处理思路。


In [ ]:
# 场景：当缺失值具有明确的业务含义时，例如用户年龄缺失可能表示用户未填写，而不是不知道年龄
# 处理思路：可以将缺失值标记为单独的类别或使用众数填充，或者根据其他相关字段进行推断


---
## 4. 类别字段标准化

同一业务含义被写成不同文本，会导致统计结果被拆散。先观察，再统一；不要在没有业务依据的情况下随意合并。


### 任务 5：查看类别取值

检查登录设备、支付方式和订单品类字段，记录你发现的同义类别。


In [ ]:
category_cols = [
    "PreferredLoginDevice",
    "PreferredPaymentMode",
    "PreferedOrderCat",
]

for col in category_cols:
    print(f"\n{col}")
    print(df[col].value_counts())


### 任务 6：统一同义类别

按以下规则进行标准化：

- Phone → Mobile Phone
- COD → Cash on Delivery
- CC → Credit Card
- Mobile → Mobile Phone

处理后重新输出频数。


In [ ]:
df["PreferredLoginDevice"] = df["PreferredLoginDevice"].replace(
    {"Phone": "Mobile Phone", "Mobile": "Mobile Phone"}
)

df["PreferredPaymentMode"] = df["PreferredPaymentMode"].replace(
    {"COD": "Cash on Delivery", "CC": "Credit Card"}
)

for col in category_cols:
    print(f"\n{col}")
    print(df[col].value_counts())


---
## 5. 候选异常值检查

IQR 方法只能发现候选异常值，不能直接证明数据错误。处理前必须结合业务判断。


In [ ]:
def iqr_outlier_summary(series):
    """返回数值字段的 IQR 候选异常值摘要。"""
    series = series.dropna()
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    return pd.Series({
        "Q1": q1,
        "Q3": q3,
        "下限": lower,
        "上限": upper,
        "候选异常值数量": ((series < lower) | (series > upper)).sum()
    })


### 任务 7：检查候选异常值

分别检查 WarehouseToHome、OrderCount 和 CashbackAmount。随后说明：候选异常值能否直接删除，为什么？


In [ ]:
display(iqr_outlier_summary(df["WarehouseToHome"]))
display(iqr_outlier_summary(df["OrderCount"]))
display(iqr_outlier_summary(df["CashbackAmount"]))

# 结论：候选异常值不能直接删除，因为IQR方法只是统计上的判断，这些所谓的异常值可能是真实存在的极端情况，
# 比如距离仓库很远的用户、购买量很大的用户或返现很高的用户，需要结合业务场景进一步判断。


### 任务 8：业务规则检查

统计以下不符合业务规则的记录数量：

- 使用时长小于 0；
- 仓库距离小于 0；
- 订单数小于或等于 0；
- 返现金额小于 0。


In [ ]:
rules = {
    "使用时长小于 0": (df["Tenure"] < 0).sum(),
    "仓库距离小于 0": (df["WarehouseToHome"] < 0).sum(),
    "订单数小于或等于 0": (df["OrderCount"] <= 0).sum(),
    "返现金额小于 0": (df["CashbackAmount"] < 0).sum(),
}
pd.Series(rules)


---
## 6. 清洗结果验收与导出

在导出前确认：指定数值字段不再有缺失；类别同义值已统一；输出目录存在。


In [ ]:
# 完成验收
assert df[numeric_missing_cols].isna().sum().sum() == 0, "数值字段仍有缺失值"
assert "Phone" not in df["PreferredLoginDevice"].unique(), "登录设备尚未统一"
assert "COD" not in df["PreferredPaymentMode"].unique(), "支付方式尚未统一"
assert "CC" not in df["PreferredPaymentMode"].unique(), "支付方式尚未统一"

print("数据清洗验收通过。")


### 任务 9：导出清洗后的数据

将文件导出至 output/ecommerce_customer_cleaned.csv。请确保原始数据不会被覆盖。


In [ ]:
OUTPUT_PATH = Path("output/ecommerce_customer_cleaned.csv")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

# 导出为 UTF-8-SIG 编码的 CSV 文件
df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

print(f"已导出：{OUTPUT_PATH.resolve()}")


---
## 7. 提交前自查

- [x] 已完成缺失值报告；
- [x] 已完成重复记录检查；
- [x] 已填补指定数值字段的缺失值；
- [x] 已统一登录设备、支付方式和订单品类；
- [x] 已完成候选异常值与业务规则检查；
- [x] 已导出 ecommerce_customer_cleaned.csv；
- [x] 已在关键代码处保留注释，说明处理理由。

## 课后思考

若要基于本数据预测用户流失，哪些字段可以作为特征？CustomerID 是否应该用于建模？为什么？

**回答：**

可以作为特征的字段包括：Tenure（使用时长）、WarehouseToHome（仓库距离）、HourSpendOnApp（App使用时长）、OrderCount（订单数）、CouponUsed（优惠券使用次数）、CashbackAmount（返现金额）、DaySinceLastOrder（距上次下单天数）、PreferredLoginDevice（登录设备）、PreferredPaymentMode（支付方式）、PreferedOrderCat（偏好订单品类）、SatisfactionScore（满意度）、Complain（投诉情况）等。

CustomerID 不应该用于建模，因为它只是用户的标识符，没有业务含义，不能提供任何预测价值，反而可能导致模型过拟合。
